# Notebook 02 — Data Cleaning & SQLite Loading

**BlueStock Fintech Internship — Capstone Project I**

## Objectives
- Clean all 10 datasets (type coercions, date parsing, deduplication)
- NAV: forward-fill missing values for weekends/public holidays
- Load cleaned data into `bluestock_mf.db` (star schema)
- Verify row counts for each table

In [ ]:
# Run the ETL pipeline directly (all cleaning + loading in one step)
import subprocess, sys
from pathlib import Path

script = Path('..') / 'scripts' / 'etl_pipeline.py'
result = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERRORS:', result.stderr)

In [ ]:
# Verify all tables loaded correctly
import sqlite3
import pandas as pd
from pathlib import Path

DB = Path('..') / 'data' / 'db' / 'bluestock_mf.db'
conn = sqlite3.connect(DB)

tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
print(f'Tables in DB: {tables}\n')

for t in tables:
    cnt = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t:<35}: {cnt:>10,} rows')

conn.close()

In [ ]:
# Sample queries to validate schema
conn = sqlite3.connect(DB)

print('── Top 5 NAV records:')
display(pd.read_sql('SELECT * FROM fact_nav LIMIT 5', conn))

print('\n── Funds per category:')
display(pd.read_sql('SELECT category, COUNT(*) as fund_count FROM dim_fund GROUP BY category ORDER BY fund_count DESC', conn))

print('\n── Date range in NAV:')
display(pd.read_sql('SELECT MIN(date) as earliest, MAX(date) as latest, COUNT(DISTINCT date) as trading_days FROM fact_nav', conn))

conn.close()
print('\n✓ Database validation complete.')